In [49]:
import csv
import json
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import cv2
from ultralytics import YOLO


In [50]:

# =========================================================
# YOLO26 Test on Video
# =========================================================
#
# Default paths are set for:
#   Computer_Vision/BoxingPunchDetection/test_video/task_kam4_gh199681/data/GH199681.mp4
#   Computer_Vision/BoxingPunchDetection/test_video/task_kam4_gh199681/annotations.json
# =========================================================\

# -----------------------------
# PATHS
# -----------------------------
# Replace following with your project path
PROJECT_ROOT = Path(r"C:\Users\benne\Computer_Vision\BoxingPunchDetection")

# Set this to your final model. This default points to the resumable final-like run.
FINAL_MODEL_PATH = PROJECT_ROOT / "yolo26_models" / "resume_yolo26s" / "weights" / "best.pt"

#change to test video path
VIDEO_PATH = PROJECT_ROOT / "input_video" / "task_kam4_gh199681" / "data" / "GH199681.mp4"
ANNOTATIONS_PATH = PROJECT_ROOT / "input_video" / "task_kam4_gh199681" / "annotations.json"

OUTPUT_ROOT = Path("runs")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

RUN_TAG = "YOLO26sConf65det2temp3"  # <-- EDIT THIS PER RUN

RUN_NAME = f"{VIDEO_PATH.stem}_{RUN_TAG}"
RUN_DIR = OUTPUT_ROOT / RUN_NAME

# -----------------------------
# TOGGLES
# -----------------------------
COMPARE_WITH_ANNOTATIONS = True
SAVE_ANNOTATED_VIDEO = True
SHOW_VIDEO = False
CONF = 0.65
IOU = 0.5
MAX_DET = 2
COMPARISON_IOU_THRESHOLD = 0.5
MAX_FRAMES: Optional[int] = None  # set to an int for quick testing, e.g. 500


# -----------------------------
# TEMPORAL FILTER TOGGLE
# -----------------------------
USE_TEMPORAL_FILTER = True
TEMPORAL_IOU = 0.3
TEMPORAL_WINDOW = 3
MIN_TEMPORAL_HITS = 2

# -----------------------------
# BACKGROUND SUBTRACTION TOGGLE
# -----------------------------

USE_BACKGROUND_SUBTRACTION = False
BACKGROUND_SUBTRACTION_MODE = "apply"  # choices: "apply", "mask"
BG_HISTORY = 500
BG_VAR_THRESHOLD = 16
BG_DETECT_SHADOWS = False
BG_LEARNING_RATE = -1  # -1 lets OpenCV choose automatically
BG_WARMUP_FRAMES = 30  # frames used to learn the background before writing output
BG_MORPH_KERNEL_SIZE = 5  # set to 0/1 to disable morphology
BG_MIN_CONTOUR_AREA = 0  # set >0 to remove tiny moving blobs from the mask



In [51]:

# -----------------------------
# CLASS MAPS
# -----------------------------
CLASS_ID_TO_NAME = {
    0: "left_head",
    1: "right_head",
    2: "left_body",
    3: "right_body",
    4: "left_block",
    5: "right_block",
    6: "left_miss",
    7: "right_miss",
}

POLISH_TO_NAME = {
    "Głowa lewą ręką": "left_head",
    "Głowa prawą ręką": "right_head",
    "Korpus lewą ręką": "left_body",
    "Korpus prawą ręką": "right_body",
    "Blok lewą ręką": "left_block",
    "Blok prawą ręką": "right_block",
    "Chybienie lewą ręką": "left_miss",
    "Chybienie prawą ręką": "right_miss",
}


@dataclass
class BoxRecord:
    frame: int
    cls_name: str
    x1: float
    y1: float
    x2: float
    y2: float
    conf: Optional[float] = None



In [52]:

def ensure_exists(path: Path, name: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"{name} not found: {path}")




def get_video_info(video_path: Path) -> Dict:
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise ValueError(f"Could not open video: {video_path}")

    info = {
        "fps": cap.get(cv2.CAP_PROP_FPS),
        "width": int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
        "height": int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
        "frame_count": int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
    }
    cap.release()
    return info


def clean_foreground_mask(mask, kernel_size: int, min_contour_area: int):
    if kernel_size and kernel_size > 1:
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    if min_contour_area and min_contour_area > 0:
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        filtered = mask * 0
        for contour in contours:
            if cv2.contourArea(contour) >= min_contour_area:
                cv2.drawContours(filtered, [contour], -1, 255, thickness=cv2.FILLED)
        mask = filtered

    return mask


def create_background_subtracted_video(video_path: Path, run_dir: Path) -> Tuple[Path, Dict]:
    """Create a background-subtracted copy of the input video for YOLO inference.

    Note: YOLO still receives full-size frames, so this is mainly a preprocessing
    toggle for cleaner input. End-to-end runtime can improve or worsen depending
    on the machine/model because the preprocessing step also costs time.
    """
    ensure_exists(video_path, "Video")
    run_dir.mkdir(parents=True, exist_ok=True)

    if BACKGROUND_SUBTRACTION_MODE not in {"apply", "mask"}:
        raise ValueError('BACKGROUND_SUBTRACTION_MODE must be "apply" or "mask"')

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise ValueError(f"Could not open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    output_path = run_dir / f"{video_path.stem}_bgsub_{BACKGROUND_SUBTRACTION_MODE}.mp4"
    writer = cv2.VideoWriter(
        str(output_path),
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (width, height),
    )
    if not writer.isOpened():
        cap.release()
        raise ValueError(f"Could not create video writer: {output_path}")

    subtractor = cv2.createBackgroundSubtractorMOG2(
        history=BG_HISTORY,
        varThreshold=BG_VAR_THRESHOLD,
        detectShadows=BG_DETECT_SHADOWS,
    )

    frame_idx = 0
    written_frames = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break

        raw_mask = subtractor.apply(frame, learningRate=BG_LEARNING_RATE)
        _, mask = cv2.threshold(raw_mask, 127, 255, cv2.THRESH_BINARY)
        mask = clean_foreground_mask(mask, BG_MORPH_KERNEL_SIZE, BG_MIN_CONTOUR_AREA)

        # Let the subtractor learn the scene before writing frames that YOLO sees.
        # This avoids writing the first few frames as almost entirely foreground.
        if frame_idx >= BG_WARMUP_FRAMES:
            if BACKGROUND_SUBTRACTION_MODE == "mask":
                output_frame = cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR)
            else:
                output_frame = cv2.bitwise_and(frame, frame, mask=mask)
            writer.write(output_frame)
            written_frames += 1

        frame_idx += 1
        if MAX_FRAMES is not None and frame_idx >= MAX_FRAMES:
            break

    cap.release()
    writer.release()

    summary = {
        "enabled": True,
        "input_video": str(video_path),
        "output_video": str(output_path),
        "mode": BACKGROUND_SUBTRACTION_MODE,
        "history": BG_HISTORY,
        "var_threshold": BG_VAR_THRESHOLD,
        "detect_shadows": BG_DETECT_SHADOWS,
        "learning_rate": BG_LEARNING_RATE,
        "warmup_frames": BG_WARMUP_FRAMES,
        "morph_kernel_size": BG_MORPH_KERNEL_SIZE,
        "min_contour_area": BG_MIN_CONTOUR_AREA,
        "input_frame_count": total_frames,
        "frames_read": frame_idx,
        "frames_written": written_frames,
        "fps": fps,
        "width": width,
        "height": height,
    }
    save_json(run_dir / "background_subtraction_summary.json", summary)
    print(f"Background-subtracted video saved to: {output_path}")
    return output_path, summary


def iou_xyxy(a: BoxRecord, b: BoxRecord) -> float:
    inter_x1 = max(a.x1, b.x1)
    inter_y1 = max(a.y1, b.y1)
    inter_x2 = min(a.x2, b.x2)
    inter_y2 = min(a.y2, b.y2)

    inter_w = max(0.0, inter_x2 - inter_x1)
    inter_h = max(0.0, inter_y2 - inter_y1)
    inter_area = inter_w * inter_h

    area_a = max(0.0, a.x2 - a.x1) * max(0.0, a.y2 - a.y1)
    area_b = max(0.0, b.x2 - b.x1) * max(0.0, b.y2 - b.y1)
    union = area_a + area_b - inter_area

    if union <= 0:
        return 0.0
    return inter_area / union


def save_json(path: Path, data) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)




In [53]:

def load_ground_truth(annotations_path: Path) -> Dict[int, List[BoxRecord]]:
    with open(annotations_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if isinstance(data, list):
        if not data:
            raise ValueError("annotations.json is empty")
        data = data[0]

    tracks = data.get("tracks", [])
    gt_by_frame: Dict[int, List[BoxRecord]] = {}

    for track in tracks:
        polish_label = track.get("label")
        cls_name = POLISH_TO_NAME.get(polish_label)
        if cls_name is None:
            continue

        for shape in track.get("shapes", []):
            if shape.get("type") != "rectangle":
                continue
            if shape.get("outside", False):
                continue

            points = shape.get("points", [])
            frame_idx = shape.get("frame")
            if frame_idx is None or len(points) != 4:
                continue

            x1, y1, x2, y2 = points
            gt_by_frame.setdefault(int(frame_idx), []).append(
                BoxRecord(
                    frame=int(frame_idx),
                    cls_name=cls_name,
                    x1=float(x1),
                    y1=float(y1),
                    x2=float(x2),
                    y2=float(y2),
                    conf=None,
                )
            )

    return gt_by_frame



In [54]:

def run_model(model_path: Path, video_path: Path, run_dir: Path) -> Tuple[Path, Dict[int, List[BoxRecord]], Dict]:
    run_dir.mkdir(parents=True, exist_ok=True)
    ensure_exists(model_path, "Model")
    ensure_exists(video_path, "Video")

    model = YOLO(model_path)

    print(f"Using model: {model_path}")
    print(f"Testing video: {video_path}")
    print(f"Saving outputs to: {run_dir}")

    # Ultralytics saves to project / name, so this writes directly to RUN_DIR
    predict_project = run_dir.parent
    predict_name = run_dir.name

    prediction_records: Dict[int, List[BoxRecord]] = {}
    prediction_json_frames = []
    processed_frames = 0

    results = model.predict(
        source=str(video_path),
        save=SAVE_ANNOTATED_VIDEO,
        project=str(predict_project),
        name=predict_name,
        conf=CONF,
        iou=IOU,
        max_det=MAX_DET,
        show=SHOW_VIDEO,
        stream=True,
        line_width=3,
        verbose=False,
    )

    for result in results:
        frame_idx = int(getattr(result, "frame", processed_frames))
        frame_entries = []
        frame_boxes: List[BoxRecord] = []

        if result.boxes is not None and len(result.boxes) > 0:
            xyxy = result.boxes.xyxy.cpu().tolist()
            confs = result.boxes.conf.cpu().tolist()
            clss = result.boxes.cls.cpu().tolist()

            for box_xyxy, conf, cls_id in zip(xyxy, confs, clss):
                cls_id = int(cls_id)
                cls_name = CLASS_ID_TO_NAME.get(cls_id, str(cls_id))
                x1, y1, x2, y2 = [float(v) for v in box_xyxy]

                rec = BoxRecord(
                    frame=frame_idx,
                    cls_name=cls_name,
                    x1=x1,
                    y1=y1,
                    x2=x2,
                    y2=y2,
                    conf=float(conf),
                )
                frame_boxes.append(rec)
                frame_entries.append(
                    {
                        "class_id": cls_id,
                        "class_name": cls_name,
                        "confidence": float(conf),
                        "xyxy": [x1, y1, x2, y2],
                    }
                )

        prediction_records[frame_idx] = frame_boxes
        prediction_json_frames.append(
            {
                "frame": frame_idx,
                "detections": frame_entries,
            }
        )

        processed_frames += 1
        if MAX_FRAMES is not None and processed_frames >= MAX_FRAMES:
            break

    annotated_dir = run_dir
    predicted_video = None
    if annotated_dir.exists():
        candidates = sorted(list(annotated_dir.glob("*.mp4")) + list(annotated_dir.glob("*.avi")) + list(annotated_dir.glob("*.mov")))
        if candidates:
            predicted_video = candidates[0]

    predictions_json = {
        "model_path": str(model_path),
        "video_path": str(video_path),
        "frames_processed": processed_frames,
        "confidence_threshold": CONF,
        "iou_threshold": IOU,
        "predictions": prediction_json_frames,
    }
    save_json(run_dir / "predictions.json", predictions_json)

    with open(run_dir / "predictions_flat.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["frame", "class_name", "confidence", "x1", "y1", "x2", "y2"])
        for frame_idx in sorted(prediction_records.keys()):
            for box in prediction_records[frame_idx]:
                writer.writerow([frame_idx, box.cls_name, box.conf, box.x1, box.y1, box.x2, box.y2])

    summary = {
        "predicted_video": str(predicted_video) if predicted_video else None,
        "predictions_json": str(run_dir / "predictions.json"),
        "predictions_csv": str(run_dir / "predictions_flat.csv"),
        "frames_processed": processed_frames,
    }

    return predicted_video, prediction_records, summary



In [55]:

def temporal_filter_predictions(
    pred_by_frame: Dict[int, List[BoxRecord]],
    temporal_iou: float = 0.3,
    temporal_window: int = 2,
    min_hits: int = 2,
) -> Dict[int, List[BoxRecord]]:
    filtered: Dict[int, List[BoxRecord]] = {}

    for frame_idx, preds in pred_by_frame.items():
        kept = []

        for pred in preds:
            hits = 1  # count itself

            for other_frame in range(frame_idx - temporal_window, frame_idx + temporal_window + 1):
                if other_frame == frame_idx:
                    continue

                for other in pred_by_frame.get(other_frame, []):
                    if other.cls_name != pred.cls_name:
                        continue

                    if iou_xyxy(pred, other) >= temporal_iou:
                        hits += 1
                        break

                if hits >= min_hits:
                    break

            if hits >= min_hits:
                kept.append(pred)

        filtered[frame_idx] = kept

    return filtered
    
def compare_predictions_to_gt(
    pred_by_frame: Dict[int, List[BoxRecord]],
    gt_by_frame: Dict[int, List[BoxRecord]],
    run_dir: Path,
    iou_threshold: float,
) -> Dict:
    run_dir.mkdir(parents=True, exist_ok=True)

    all_frames = sorted(set(pred_by_frame.keys()) | set(gt_by_frame.keys()))

    total_tp = 0
    total_fp = 0
    total_fn = 0
    per_frame_rows = []
    match_rows = []

    for frame_idx in all_frames:
        preds = pred_by_frame.get(frame_idx, [])
        gts = gt_by_frame.get(frame_idx, [])

        matched_pred = set()
        matched_gt = set()

        candidate_matches = []
        for pi, pred in enumerate(preds):
            for gi, gt in enumerate(gts):
                if pred.cls_name != gt.cls_name:
                    continue
                score = iou_xyxy(pred, gt)
                if score >= iou_threshold:
                    candidate_matches.append((score, pi, gi))

        candidate_matches.sort(reverse=True, key=lambda x: x[0])

        for score, pi, gi in candidate_matches:
            if pi in matched_pred or gi in matched_gt:
                continue
            matched_pred.add(pi)
            matched_gt.add(gi)
            total_tp += 1
            match_rows.append([
                frame_idx,
                preds[pi].cls_name,
                round(score, 6),
                preds[pi].conf,
                preds[pi].x1,
                preds[pi].y1,
                preds[pi].x2,
                preds[pi].y2,
                gts[gi].x1,
                gts[gi].y1,
                gts[gi].x2,
                gts[gi].y2,
            ])

        frame_fp = len(preds) - len(matched_pred)
        frame_fn = len(gts) - len(matched_gt)
        total_fp += frame_fp
        total_fn += frame_fn

        per_frame_rows.append([
            frame_idx,
            len(preds),
            len(gts),
            len(matched_pred),
            frame_fp,
            frame_fn,
        ])

    precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
    recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    with open(run_dir / "comparison_per_frame.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["frame", "num_predictions", "num_ground_truth", "true_positives", "false_positives", "false_negatives"])
        writer.writerows(per_frame_rows)

    with open(run_dir / "matched_detections.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            "frame",
            "class_name",
            "iou",
            "pred_confidence",
            "pred_x1", "pred_y1", "pred_x2", "pred_y2",
            "gt_x1", "gt_y1", "gt_x2", "gt_y2",
        ])
        writer.writerows(match_rows)

    summary = {
        "iou_threshold": iou_threshold,
        "true_positives": total_tp,
        "false_positives": total_fp,
        "false_negatives": total_fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "comparison_per_frame_csv": str(run_dir / "comparison_per_frame.csv"),
        "matched_detections_csv": str(run_dir / "matched_detections.csv"),
    }
    save_json(run_dir / "comparison_summary.json", summary)
    return summary



In [56]:

def main() -> None:
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Saving to: runs/{RUN_TAG}")
    inference_video_path = VIDEO_PATH
    background_subtraction_summary = {"enabled": False}

    if USE_BACKGROUND_SUBTRACTION:
        inference_video_path, background_subtraction_summary = create_background_subtracted_video(
            video_path=VIDEO_PATH,
            run_dir=RUN_DIR,
        )

    predicted_video, pred_by_frame, pred_summary = run_model(
        model_path=FINAL_MODEL_PATH,
        video_path=inference_video_path,
        run_dir=RUN_DIR,
    )
    raw_count = sum(len(v) for v in pred_by_frame.values())

    if USE_TEMPORAL_FILTER:
        pred_by_frame = temporal_filter_predictions(
            pred_by_frame,
            temporal_iou=TEMPORAL_IOU,
            temporal_window=TEMPORAL_WINDOW,
            min_hits=MIN_TEMPORAL_HITS,
        )

    filtered_count = sum(len(v) for v in pred_by_frame.values())

    print(f"Raw predictions: {raw_count}")
    print(f"After temporal filter: {filtered_count}")
    final_summary = {
        "model_path": str(FINAL_MODEL_PATH),
        "original_video_path": str(VIDEO_PATH),
        "inference_video_path": str(inference_video_path),
        "background_subtraction": background_subtraction_summary,
        "compare_with_annotations": COMPARE_WITH_ANNOTATIONS,
        "prediction_outputs": pred_summary,
    }

    if COMPARE_WITH_ANNOTATIONS:
        ensure_exists(ANNOTATIONS_PATH, "Annotations")
        gt_by_frame = load_ground_truth(ANNOTATIONS_PATH)
        comparison_summary = compare_predictions_to_gt(
            pred_by_frame=pred_by_frame,
            gt_by_frame=gt_by_frame,
            run_dir=RUN_DIR,
            iou_threshold=COMPARISON_IOU_THRESHOLD,
        ) 
        final_summary["comparison_outputs"] = comparison_summary
        print("Comparison complete.")
        print(json.dumps(comparison_summary, indent=2))
    else:
        print("Comparison skipped because COMPARE_WITH_ANNOTATIONS = False")
    save_json(RUN_DIR / "run_summary.json", final_summary)

    print("\nDone.")
    if predicted_video is not None:
        print(f"Annotated video: {predicted_video}")
    print(f"Inference video used: {inference_video_path}")
    if USE_BACKGROUND_SUBTRACTION:
        print(f"Background subtraction summary: {RUN_DIR / 'background_subtraction_summary.json'}")
    print(f"Predictions JSON: {RUN_DIR / 'predictions.json'}")
    print(f"Run summary: {RUN_DIR / 'run_summary.json'}")


if __name__ == "__main__":
    main()

Saving to: runs/YOLO26sConf65det2temp3
Using model: C:\Users\benne\Computer_Vision\BoxingPunchDetection\yolo26_models\resume_yolo26s\weights\best.pt
Testing video: C:\Users\benne\Computer_Vision\BoxingPunchDetection\input_video\task_kam4_gh199681\data\GH199681.mp4
Saving outputs to: runs\GH199681_YOLO26sConf65det2temp3
Results saved to C:\Users\benne\Computer_Vision\BoxingPunchDetection\runs\detect\runs\GH199681_YOLO26sConf65det2temp3
Raw predictions: 1310
After temporal filter: 1159
Comparison complete.
{
  "iou_threshold": 0.5,
  "true_positives": 197,
  "false_positives": 962,
  "false_negatives": 411,
  "precision": 0.16997411561691114,
  "recall": 0.32401315789473684,
  "f1": 0.22297679683078667,
  "comparison_per_frame_csv": "runs\\GH199681_YOLO26sConf65det2temp3\\comparison_per_frame.csv",
  "matched_detections_csv": "runs\\GH199681_YOLO26sConf65det2temp3\\matched_detections.csv"
}

Done.
Inference video used: C:\Users\benne\Computer_Vision\BoxingPunchDetection\input_video\task_